In [ ]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(".."))

# Розширений список стоп-слів для українських новин
UA_STOP_WORDS = [
    'та', 'на', 'що', 'це', 'як', 'для', 'до', 'про', 'він', 'за', 'з', 'і', 'в', 'у',
    'його', 'її', 'вони', 'було', 'буде', 'які', 'який', 'яка', 'або', 'але', 'чи',
    'року', 'років', 'сьогодні', 'також', 'після', 'вже', 'ще', 'час', 'заявив', 
    'повідомив', 'від', 'тільки', 'щоб', 'тому', 'бути', 'був', 'була', 'коли',
    'сказав', 'під', 'через', 'цього', 'того', 'тим', 'свій', 'свою', 'свої', 'я', 'ми',
    'не', 'а', 'же', 'ж'
]

RANDOM_STATE = 15

In [2]:
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/nlp'):
        !git clone -b lab-08-branch --single-branch https://github.com/jaYulichka46/nlp.git
    
    %cd /content/nlp
    !pip install pandas numpy scikit-learn matplotlib seaborn -q
    sys.path.append('/content/nlp')

    FOLDER_ID = '1Xhu4xjZpRu-RP730-hyErp5F0C3l_EvO'
    
    os.makedirs('data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O data/
    
    data_dir = 'data/processed_v2'
else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data/processed_v2'

In [4]:
### 2. Data access
data_file = os.path.join(data_dir, 'processed_v_2.csv')
df = pd.read_csv(data_file)

text_col = 'processed_text' if 'processed_text' in df.columns else 'clean_text'

df = df.dropna(subset=[text_col])

In [ ]:
### 3. Corpus filtering / preprocessing checks
from src.topic_modeling import build_lsa_pipeline, build_lda_pipeline
from src.topic_utils import print_top_words, print_top_documents

def count_words(text):
    return len(str(text).split())

In [6]:
df['word_count'] = df[text_col].apply(count_words)
df_filtered = df[df['word_count'] >= 10].copy()
texts = df_filtered[text_col].tolist()

print(f"Документів ДО фільтрації: {len(df)}")
print(f"Документів ПІСЛЯ фільтрації (>=10 слів): {len(texts)}")

### 4. Vectorizer setup
MIN_DF = 5
MAX_DF = 0.85

print(f"\nБазові параметри для Векторизаторів:")
print(f"min_df = {MIN_DF}")
print(f"max_df = {MAX_DF}")
print(f"stop_words = UA_STOP_WORDS ({len(UA_STOP_WORDS)} слів)")

Документів ДО фільтрації: 120417
Документів ПІСЛЯ фільтрації (>=10 слів): 120290

Базові параметри для Векторизаторів:
min_df = 5
max_df = 0.85
stop_words = UA_STOP_WORDS (58 слів)


In [7]:
### 5. LSA experiments
import time

print("Навчання моделей LSA (TF-IDF + TruncatedSVD)")
start_time = time.time()

# Варіант 1: LSA, k=5
lsa_5 = build_lsa_pipeline(
    n_components=5, min_df=MIN_DF, max_df=MAX_DF, 
    stop_words=UA_STOP_WORDS, random_state=RANDOM_STATE
)
doc_topic_lsa_5 = lsa_5.fit_transform(texts)
print("LSA (k=5) успішно навчено!")

# Варіант 2: LSA, k=8
lsa_8 = build_lsa_pipeline(
    n_components=8, min_df=MIN_DF, max_df=MAX_DF, 
    stop_words=UA_STOP_WORDS, random_state=RANDOM_STATE
)
doc_topic_lsa_8 = lsa_8.fit_transform(texts)
print("LSA (k=8) успішно навчено!")

print(f"Час навчання LSA: {time.time() - start_time:.2f} секунд")

Навчання моделей LSA (TF-IDF + TruncatedSVD)
LSA (k=5) успішно навчено!
LSA (k=8) успішно навчено!
Час навчання LSA: 46.49 секунд


In [8]:
### 6. LDA experiments
print("Навчання моделей LDA (CountVectorizer + LatentDirichletAllocation)")
start_time = time.time()

# Варіант 3: LDA, k=5
lda_5 = build_lda_pipeline(
    n_components=5, min_df=MIN_DF, max_df=MAX_DF, 
    stop_words=UA_STOP_WORDS, random_state=RANDOM_STATE
)
doc_topic_lda_5 = lda_5.fit_transform(texts)
print("LDA (k=5) успішно навчено!")

# Варіант 4: LDA, k=8
lda_8 = build_lda_pipeline(
    n_components=8, min_df=MIN_DF, max_df=MAX_DF, 
    stop_words=UA_STOP_WORDS, random_state=RANDOM_STATE
)
doc_topic_lda_8 = lda_8.fit_transform(texts)
print("LDA (k=8) успішно навчено!")

print(f"Час навчання LDA: {time.time() - start_time:.2f} секунд")

Навчання моделей LDA (CountVectorizer + LatentDirichletAllocation)
LDA (k=5) успішно навчено!
LDA (k=8) успішно навчено!
Час навчання LDA: 1121.92 секунд


### Обґрунтування вибору представлення тексту (відповідно до вимог)

Для обох моделей (LSA та LDA) ми свідомо використали:
* `analyzer="word"`
* `ngram_range=(1,1)` (уніграми)
* `min_df=5`, `max_df=0.85`

**Чому саме так:**
1. **Word features vs char-ngrams:** Для тематичного моделювання (Topic Modeling) завжди критично важлива інтерпретованість. На відміну від задач класифікації (де char-ngrams допомагають ловити опечатки), тут `char-ngrams` згенерували б абсолютно нечитабельні "шматки" слів замість тем.
2. **Уніграми `(1,1)`:** Ми почали з базових слів, оскільки для новинного корпусу окремих іменників та дієслів часто достатньо для розуміння сюжету (напр., "матч", "суд", "бюджет"). Біграми `(1,2)` ми поки що не включали, щоб не роздувати словник і зберегти максимальну простоту ручної інтерпретації на базовому рівні.

In [9]:
### 7. Top words per topic
print("ТОП-10 СЛІВ ДЛЯ LSA (k=5)\n" + "="*40)
lsa_model = lsa_5.named_steps['lsa']
lsa_vec = lsa_5.named_steps['vectorizer']
print_top_words(lsa_model, lsa_vec.get_feature_names_out(), n_top_words=10)

print("\nТОП-10 СЛІВ ДЛЯ LDA (k=5)\n" + "="*40)
lda_model = lda_5.named_steps['lda']
lda_vec = lda_5.named_steps['vectorizer']
print_top_words(lda_model, lda_vec.get_feature_names_out(), n_top_words=10)

ТОП-10 СЛІВ ДЛЯ LSA (k=5)
Topic #0: україни грн по повідомляє із україні сша так якщо еспресо
Topic #1: динамо матч ліги матчі туру го ставку матчу чемпіонату чемпіонів
Topic #2: грн курс євро долара гривні 27 рівні млрд продажу 28
Topic #3: ато еспресо tv сил районі україни прес гранатометів зброї стрілецької
Topic #4: україни росії президент ради президента щодо україна єс міністр прем


ТОП-10 СЛІВ ДЛЯ LDA (k=5)
Topic #0: повідомляє еспресо по україни tv прес області із йдеться районі
Topic #1: якщо може так україни можна їх із україні можуть те
Topic #2: динамо матчі дуже матч україни все ліги команди по так
Topic #3: україни повідомляє сша нагадаємо еспресо tv росії із президент президента
Topic #4: грн млрд млн компанії році україни ринку 10 2020 компанія



In [10]:
### 8. Top documents per topic
print("ТОП-ДОКУМЕНТИ ДЛЯ LSA (k=5)\n" + "="*50)
print_top_documents(doc_topic_lsa_5, texts, n_top_docs=2)

print("\nТОП-ДОКУМЕНТИ ДЛЯ LDA (k=5)\n" + "="*50)
print_top_documents(doc_topic_lda_5, texts, n_top_docs=2)

ТОП-ДОКУМЕНТИ ДЛЯ LSA (k=5)
Найкращі документи для Теми #0
[1] (Вага: 0.407): "У першу чергу я фінансист, а не політик, тому і залишилася в уряді", — говорить міністр фінансів Оксана Маркарова у розмові з ЕП. Це вже третій уряд, у якому працює Маркарова. У міністерство вона прийшла в березні 2015 року на посаду заступниці міністра фінансів Наталії Яресько в уряд Арсенія Яценюка. За ці роки працювала на посадах першої заступниці міністра, виконувачки обов’язки міністра. У 20...
[2] (Вага: 0.405): Успіх Володимра Зеленського і "Слуги народу" на виборах зробив цих людей відповідальними за те, що буде відбуватись в Україні у найближчі п’ять років. Напередодні УП уже писала, яку кандидатуру прем’єр-міністра Зеленський може внести на голосування нового парламенту вже у вересні. Очевидних фаворитів тут немає. Так само немає відповіді на питання, хто очолить міністерства і скільки взагалі їх тепе...
--------------------------------------------------
Найкращі документи для Теми #1
[1] (Вага: 0.

## Секція 5: Ручна інтерпретація тем (LSA vs LDA)

### Інтерпретація тем LSA (TF-IDF + SVD)

**LSA Тема #1: Спорт (Футбол)**
* **Топ-слова:** `динамо`, `матч`, `ліги`, `матчі`, `туру`, `чемпіонату`, `чемпіонів`.
* **Пояснення:** Ідеально чиста і сформована тема, присвячена спортивним новинам, зокрема футболу. Топ-слова чітко вказують на турніри (Ліга чемпіонів, чемпіонат) та команди (Динамо). Топ-документи повністю це підтверджують: обидва тексти є анонсами футбольних матчів ("Манчестер Сіті - Шахтар" та "Динамо - Лугано").

**LSA Тема #2: Фінанси та Валютний ринок**
* **Топ-слова:** `грн`, `курс`, `євро`, `долара`, `гривні`, `27`, `рівні`, `млрд`.
* **Пояснення:** Дуже якісна тема, сфокусована суто на фінансах, зокрема на коливаннях курсу валют. Наявність цифр `27`, `28` у топ-словах (це курс долара в Україні в той період) є чудовим індикатором специфіки новин. Документи — це класичні зведення з міжбанківського валютного ринку від Мінфіну.

**LSA Тема #3: Війна та ООС (АТО)**
* **Топ-слова:** `ато`, `сил`, `районі`, `прес`, `гранатометів`, `зброї`, `стрілецької`.
* **Пояснення:** Вузька і дуже точна тема, що описує щоденні фронтові зведення з зони бойових дій на сході України. Слова "сил", "районі", "прес" (від прес-центр) та назви зброї чітко формують цей сюжет. Обидва документи — це офіційні ранкові чи вечірні зведення від прес-центру Операції об'єднаних сил.

**LSA Тема #4: Внутрішня та Зовнішня Політика**
* **Топ-слова:** `україни`, `росії`, `президент`, `ради`, `президента`, `щодо`, `єс`, `міністр`.
* **Пояснення:** Тема згрупувала новини про вищі ешелони влади. Слова "президент", "прем'єр", "рада" вказують на офіційні заяви політиків. Топ-документи (заява Яценюка про санкції проти РФ та указ Зеленського про інвестиційну раду) підтверджують політичний та управлінський характер теми.

---

### Інтерпретація тем LDA (Count + LatentDirichletAllocation)

**LDA Тема #2: Спорт загалом**
* **Топ-слова:** `динамо`, `матчі`, `дуже`, `матч`, `україни`, `все`, `ліги`, `команди`.
* **Пояснення:** Аналог спортивної теми з LSA, але тут вона вийшла трохи ширшою і охоплює не лише футбол (у документах ми бачимо новини про стрибки з трампліна на Чемпіонаті світу). Вона менш "словникова", але контекстуально правильна.

**LDA Тема #3: Міжнародна політика (США-РФ-Україна)**
* **Топ-слова:** `україни`, `сша`, `росії`, `президент`, `президента`.
* **Пояснення:** Тут LDA вдало згрупувала політичні новини міжнародного масштабу. Документи фокусуються на стосунках США, Росії та України (зокрема, згадуються Трамп, Путін та скандал з телефонною розмовою, який призвів до спроби імпічменту Трампа).

**LDA Тема #4: Бізнес, Економіка та Банкінг**
* **Топ-слова:** `грн`, `млрд`, `млн`, `компанії`, `році`, `україни`, `ринку`, `компанія`.
* **Пояснення:** Більш широка економічна тема, ніж валютна в LSA. Вона охоплює і курс валют (як у першому документі), і прибутки корпорацій/банків (як у документі про ПриватБанк). Дуже сильна бізнес-тема.

## Секція 6: Аналіз "Поганих тем"

Незважаючи на успішне виділення рубрик спорту, фінансів та політики, моделі також згенерували слабкі кластери, які не несуть бізнес-цінності.

### Погана Тема 1: "Stop-word / Too generic topic" (LDA Тема #1)
* **Топ-слова:** `якщо`, `може`, `так`, `україни`, `можна`, `їх`, `із`, `україні`, `можуть`, `те`.
* **6.1. Чому тема слабка:** Це класична сміттєва тема, що складається виключно з часток, займенників та модальних дієслів. Вона абсолютно загальна, не прив'язана до жодного новинного сюжету і відображає лише граматичну структуру українських речень.
* **6.2. Чому тема вийшла поганою:** Головна причина — **проблема в стоп-словах**. Незважаючи на те, що ми передали кастомний словник `UA_STOP_WORDS`, українська мова дуже багата на загальні слова. Такі токени як *"якщо", "може", "їх", "те"* прослизнули крізь фільтр. Друга причина — **проблема в max_df**: ці слова є майже в кожній розгорнутій новині, але поріг `0.85` виявився для них занадто м'яким.
* **6.3. Що б ми змінили далі:** 1. **Покращити stop-word filtering:** Додати всі ці виявлені модальні слова та займенники у наш кастомний масив стоп-слів.
  2. **Змінити max_df:** Знизити поріг `max_df` з `0.85` до `0.60` або `0.50`, щоб автоматично "зрізати" слова, які розпорошені по всьому корпусу.

---

### Погана Тема 2: "Template / Domain noise topic" (LDA Тема #0)
* **Топ-слова:** `повідомляє`, `еспресо`, `по`, `україни`, `tv`, `прес`, `області`, `із`, `йдеться`, `районі`.
* **6.1. Чому тема слабка:** Модель вловила не суть новини, а шаблонний стиль та технічний шум конкретного медіа. Слова *"повідомляє", "йдеться", "прес"* — це журналістські штампи, а *"еспресо tv"* — маркер джерела парсингу.
* **6.2. Чому тема вийшла поганою:** Це пряма **проблема шаблонної лексики та неоднорідності корпусу**. Очевидно, велика частка датасету була спарсена з ресурсу "Еспресо TV", і в текстах залишилися підписи редакції чи копірайт. Алгоритм побачив, що слова "еспресо" і "tv" часто зустрічаються разом із "повідомляє", і виділив це як окремий "сюжет".
* **6.3. Що б ми змінили далі:**
  1. **Покращити preprocessing:** На етапі регулярних виразів (до векторизації) жорстко вирізати назви новинних агенцій, копірайти та штампи типу *"Як повідомляє прес-служба"*.
  2. **Додати bigrams (ngram_range=(1,2)):** Якщо дозволити біграми, фраза "еспресо tv" склеїться в один токен, який потім буде набагато легше занести у stop-words як єдиний об'єкт доменного шуму.

## Секція 7: Битва моделей — LSA vs LDA

### 7.1. Яка модель дала більш читабельні теми?
У нашому експерименті **LSA дала значно більш читабельні та чіткі теми**. Її кластери виглядають як ідеальні теги для новин (Спорт, Фінанси/Валюта, Війна/АТО). LDA також знайшла осмислені сюжети, але вони вийшли більш розмитими та вимагали глибшого вчитування в документи для розуміння контексту.

### 7.2. Аналіз недоліків (шум, дублікати, змішування)
* **Шумних тем більше у LDA.** Модель LDA витратила цілих дві теми (Тема #0 та Тема #1) на групування українських стоп-слів та шаблонного медійного шуму ("еспресо tv", "повідомляє", "якщо", "може").
* **Змішаних тем також більше у LDA.** У той час як LSA чітко відділила внутрішню політику, LDA схильна створювати ширші "кошики" (наприклад, поєднала бізнес, компанії та загальну економіку в один великий кластер).
* **Дубльованих тем** при $k=5$ майже не було в обох моделях, оскільки кількість тем замала для їх розщеплення, але LDA була ближчою до цього через загальну лексику.

### 7.3. Яка модель корисніша саме для корпусу UA News?
Для нашого датасету українських новин **LSA виявилася значно кориснішою та стабільнішою**. Оскільки новини містять багато специфічних маркерів (прізвища політиків, назви команд, терміни на кшталт "АТО" чи "міжбанк"), використання TF-IDF у LSA допомогло "приглушити" загальновживану лексику і витягнути наверх саме суть події. LDA, спираючись на звичайний CountVectorizer, виявилася занадто чутливою до специфічного журналістського стилю та маркерів парсингу ("Еспресо TV", "прес-служба"). Через це імовірнісний алгоритм LDA витратив свій ресурс на кластеризацію граматичних конструкцій і штампів, а не реальних новинних сюжетів. Отже, без ультражорсткої попередньої чистки журналістських шаблонів, LSA є надійнішим вибором для цього корпусу.

## Секція 8: Роль метрики Coherence (Узгодженість тем)

У тематичному моделюванні часто використовують метрику **Topic Coherence** для автоматичного підбору оптимальної кількості тем ($k$). Вона оцінює, наскільки часто топові слова теми зустрічаються разом у реальних документах корпусу.

Проте в цій лабораторній роботі ми свідомо зробили головний акцент саме на **ручній оцінці та аналізі топ-документів**, керуючись наступною логікою:

**Чому Coherence не замінює ручну інтерпретацію (на прикладі UA News):**
1. **Ілюзія якості на шумі:** Наша "погана" тема з LDA (Тема #0), яка зібрала слова *"повідомляє", "еспресо", "tv", "прес"*, математично могла б отримати дуже високий бал Coherence. Чому? Бо ці слова майже завжди стоять поруч у копірайтах та журналістських шаблонах цього видання. Алгоритм вважав би цю тему "ідеально узгодженою", хоча її реальна інформаційна та бізнес-цінність дорівнює нулю.
2. **Стоп-слова:** Тема з модальними словами та займенниками (*"якщо", "може", "так", "можна"*) також може мати високу узгодженість через базову структуру української граматики.

**Висновок:** Coherence — це корисний додатковий сигнал, який допомагає звузити діапазон пошуку $k$ (наприклад, між 5 і 50). Але єдиною надійною основою для фінального висновку щодо якості моделі завжди залишається **ручна оцінка** (перевірка того, чи формують топ-слова логічний сюжет) та **читання top documents** (перевірка того, чи відповідає текст цьому сюжету на практиці).

In [11]:
### 12. Generate docs/audit_summary_lab8.md

os.makedirs("../docs", exist_ok=True)

audit_text = """# Audit Summary: Lab 8 (Topic Modeling)

1. **Завдання:** Unsupervised clustering тексту для виявлення прихованих новинних рубрик в датасеті UA News.
2. **Дані та Фільтрація:** Використано українські новини (>= 10 слів). Параметри векторизаторів: `min_df=5`, `max_df=0.85`, кастомний список `UA_STOP_WORDS`.
3. **Моделі:** * LSA (TF-IDF + TruncatedSVD)
   * LDA (CountVectorizer + LatentDirichletAllocation)
   * Експерименти проведено для $k=5$ та $k=8$.
4. **Успішні теми:** LSA ідеально виділила класичні рубрики: **Спорт (Футбол)**, **Фінанси (Валюта)**, **Війна (АТО)** та **Політика**.
5. **Погані теми (Аналіз):**
   * **Stop-word topic:** LDA виділила загальну тему з модальних слів та займенників (*якщо, може, так, можна*). Причина: багатство української мови на такі слова та надто м'який поріг `max_df=0.85`.
   * **Domain noise topic:** LDA створила тему навколо журналістських штампів та джерел (*повідомляє, еспресо, tv, прес*). Причина: специфіка парсингу конкретних медіа.
6. **LSA vs LDA:** Для нашого новинного датасету **LSA виявилася значно читабельнішою та точнішою**. Вона змогла приглушити загальну лексику завдяки TF-IDF, тоді як імовірнісна LDA витратила ресурс на моделювання журналістських шаблонів та копірайтів.
7. **Coherence:** Метрика Coherence не використовувалась як основний критерій, оскільки для новинного корпусу тема з чистого доменного шуму ("еспресо tv") може мати високий бал узгодженості через часту сумісність цих слів, але не мати жодної бізнес-цінності.
"""

with open("../docs/audit_summary_lab8.md", "w", encoding="utf-8") as f:
    f.write(audit_text)

print("Файл docs/audit_summary_lab8.md успішно згенеровано!")

Файл docs/audit_summary_lab8.md успішно згенеровано!
